# LIGER default - human_pancreas

Seeds: `1-10`

Metrics: BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR


In [1]:
import gc
import importlib.util
import os
import sys

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics'
os.environ["R_HOME"] = f"{PROJECT_ROOT}/CondaEnv/pyliger_env/lib/R"
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import numpy as np
import pandas as pd

OPTIMIZER_PATH = (
    f"{PROJECT_ROOT}/scripts/optuna_multi_metrics/"
    "optuna_liger/optimize_liger_multi_metrics.py"
)
optimizer_spec = importlib.util.spec_from_file_location(
    "liger_opt",
    OPTIMIZER_PATH,
)
if optimizer_spec is None or optimizer_spec.loader is None:
    raise ImportError(f"Cannot load LIGER optimizer: {OPTIMIZER_PATH}")
liger_opt = importlib.util.module_from_spec(optimizer_spec)
sys.modules[optimizer_spec.name] = liger_opt
optimizer_spec.loader.exec_module(liger_opt)


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_NAME = 'human_pancreas'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'tech'
LABEL_KEY = 'celltype'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

SEEDS = tuple(range(1, 11))
MODE = "core"
LAYER = "counts"
N_NEIGHBORS = 15
EMBED_KEY = "X_liger"

liger_opt.DATASET_NAME = DATASET_NAME
liger_opt.DATA_PATH = DATA_PATH
liger_opt.BATCH_KEY = BATCH_KEY
liger_opt.LABEL_KEY = LABEL_KEY
liger_opt.MODE = MODE
liger_opt.LAYER = LAYER
liger_opt.USE_BAMI = "BAMI" in SELECTED_METRICS
liger_opt.USE_CASW = "CASW" in SELECTED_METRICS
liger_opt.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
liger_opt.USE_TRAJ = "TRAJ" in SELECTED_METRICS
liger_opt.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
liger_opt.USE_BRAS = "BRAS" in SELECTED_METRICS
liger_opt.USE_KBET = "KBET" in SELECTED_METRICS
liger_opt.USE_CPCR = "CPCR" in SELECTED_METRICS

assert DATASET_NAME == 'human_pancreas'
assert BATCH_KEY == 'tech'
assert LABEL_KEY == 'celltype'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI', 'BRAS', 'KBET', 'CPCR')
assert SEEDS == tuple(range(1, 11))
assert liger_opt.CORE_MAX_ITERS == 30
assert liger_opt.N_NEIGHBORS == N_NEIGHBORS
assert liger_opt.EMBED_KEY == EMBED_KEY

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(SEEDS),
        "mode": MODE,
        "layer": LAYER,
    },
    name="configuration",
)


dataset                                         human_pancreas
data_path    /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                                 tech
label_key                                             celltype
metrics      [BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR]
seeds                          [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
mode                                                      core
layer                                                   counts
Name: configuration, dtype: object

In [3]:
liger_opt.validate_metric_dependencies()
liger_opt.validate_configuration(list(SELECTED_METRICS))
liger_opt.validate_dataset_contents()


Dataset preflight: n_obs=16382 n_batches=9 n_labels=14
ISOLISI preflight: min_batch_coverage=4 isolated_labels=['t_cell']
CPCR preflight: eligible_labels=9


In [4]:
LIGER_DEFAULT_ARGS = {
    "k": 20,
    "value_lambda": 5.0,
    "max_iters": 30,
    "mode": "core",
    "layer": "counts",
}


def run_liger_default(adata, seed):
    return liger_opt.run_liger(
        adata=adata,
        k=LIGER_DEFAULT_ARGS["k"],
        lam=LIGER_DEFAULT_ARGS["value_lambda"],
        max_iterations=LIGER_DEFAULT_ARGS["max_iters"],
        mode=LIGER_DEFAULT_ARGS["mode"],
        layer=LIGER_DEFAULT_ARGS["layer"],
        seed=seed,
    )


pd.Series(LIGER_DEFAULT_ARGS, name="default_setting")


k                   20
value_lambda       5.0
max_iters           30
mode              core
layer           counts
Name: default_setting, dtype: object

In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    return liger_opt.compute_joint_metrics(adata_raw, adata_int)


In [6]:
results = []

for seed in SEEDS:
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_liger_default(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 1 =====


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the

Removing 2011 genes not expressing in celseq.
Removing 2693 genes not expressing in celseq2.
Removing 2724 genes not expressing in fluidigmc1.


Removing 3518 genes not expressing in inDrop1.
Removing 3238 genes not expressing in inDrop2.
Removing 2910 genes not expressing in inDrop3.
Removing 3788 genes not expressing in inDrop4.


Removing 2032 genes not expressing in smarter.
Removing 1286 genes not expressing in smartseq2.


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.6428729880954557


optimised clustering against liger_metric_label
optimal cluster resolution: 0.4
optimal score: 0.6845312936503195
BRAS...


seed           10.000000
BAMI            0.684531
CASW            0.609103
SCGRAPHM        0.388067
ISOLISI         0.821953
BRAS            0.743364
KBET            0.638016
CPCR            0.855375
bio_score       0.625914
batch_score     0.745585
joint_score     0.673782
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,ISOLISI,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,1,0.702451,0.621262,0.467011,0.869113,0.719616,0.632439,0.887774,0.664959,0.746610,0.697619
1,2,0.715295,0.651889,0.424300,0.743970,0.700741,0.644083,0.877965,0.633864,0.740930,0.676690
2,3,0.703411,0.630812,0.539123,0.888613,0.721760,0.638021,0.879104,0.690490,0.746295,0.712812
3,4,0.683879,0.619423,0.485865,0.848413,0.710803,0.665899,0.867949,0.659395,0.748217,0.694924
4,5,0.724227,0.662620,0.418117,0.962172,0.710237,0.624637,0.880651,0.691784,0.738508,0.710474
5,6,0.706394,0.634714,0.411769,0.924355,0.714133,0.618132,0.873926,0.669308,0.735397,0.695744
6,7,0.660681,0.649574,0.443274,0.903313,0.701237,0.613125,0.857903,0.664211,0.724088,0.688162
7,8,0.679972,0.648844,0.407732,0.957099,0.671626,0.653842,0.862366,0.673412,0.729278,0.695758
8,9,0.712372,0.627279,0.409365,0.948630,0.746140,0.637190,0.902280,0.674412,0.761870,0.709395
9,10,0.684531,0.609103,0.388067,0.821953,0.743364,0.638016,0.855375,0.625914,0.745585,0.673782


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.697321,0.018458,0.660681,0.724227
CASW,0.635552,0.016212,0.609103,0.662620
SCGRAPHM,0.439462,0.043434,0.388067,0.539123
ISOLISI,0.886763,0.065210,0.743970,0.962172
BRAS,0.713966,0.020361,0.671626,0.746140
KBET,0.636538,0.015062,0.613125,0.665899
CPCR,0.874529,0.013604,0.855375,0.902280
bio_score,0.664775,0.020184,0.625914,0.691784
batch_score,0.741678,0.010122,0.724088,0.761870
joint_score,0.695536,0.012654,0.673782,0.712812
